# 2.1 $\quad$ Portfolio Generation

## 1. Return and volatility of a two-asset portfolio

A portfolio is a collection of financial assets, such as stocks, bonds, and other investment instruments. Its composition is chosen with the objective of achieving a specific financial goal, such as maximizing return, minimizing risk, or finding an appropriate balance between the two. Portfolio management involves making strategic decisions about asset allocation, diversification, and periodic adjustments to adapt to possible changes in market conditions.

Defining a portfolio requires specifying the assets included in it and the proportion invested in each one. Let:

* $w_1$ be the proportion invested in asset 1 $\Rightarrow w_2=1-w_1$. We must have $w_1+w_2=1.0$, since the weights represent 100% of the investment.

* $\bar R_p$ be the mean or expected return of the portfolio:
  $\bar R_p=w_1\bar R_1+w_2\bar R_2$.
  It is a weighted average of the return of each asset.

* Portfolio variance:
  $\sigma_P^2=w_1^2\sigma_1^2+w_2^2\sigma_2^2+2w_1w_2\sigma_{12}$,
  where $\sigma_{12}$ is the covariance between assets 1 and 2.

* Portfolio volatility:
  $\sigma_P=\sqrt{\sigma_P^2}$

Writing covariance as a function of correlation:

$Corr(r_1,r_2)=\frac{Cov(r_1,r_2)}{\sigma_{r_1}\sigma_{r_2}}$

so:

$Cov(r_1,r_2)=\sigma_1\sigma_2\rho_{12}$

Therefore:

$\sigma_P=\sqrt{w_1^2\sigma_1^2+w_2^2\sigma_2^2+2w_1w_2\sigma_1\sigma_2\rho_{12}}$

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import plotly.graph_objects as go

We define two functions:

* **calc_portfolio_return**: given a portfolio, it returns the expected return.

* **calc_portfolio_std**: given a portfolio, it returns the portfolio volatility.

* We also define an example portfolio with two assets.

In [ ]:
def calc_portfolio_return(w1, w2, r1, r2):
    return w1 * r1 + w2 * r2


In [ ]:
def calc_portfolio_std(w1, w2, std1, std2, rho):
    return np.sqrt(w1**2 * std1**2 + w2**2 * std2**2 + 2 * w1 * w2 * std1 * std2 * rho)


In [ ]:
# Define a portfolio with two assets
ret1 = 0.2  # expected return of asset 1
ret2 = 0.1  # expected return of asset 2

std1 = 0.25  # volatility of asset 1
std2 = 0.15  # volatility of asset 2

p1 = 0.4  # weight of asset 1
p2 = 0.6  # weight of asset 2


---

### Case 1:  
$\quad \rho_{12}=1$

* If $\rho_{12}=1$, the assets are **perfectly positively correlated**.

    - There are no diversification benefits.

    - Portfolio return is equal to the weighted sum of the asset returns:
      $E(\tilde R_P)=w_1E(\tilde R_1)+w_2E(\tilde R_2)$

    - Portfolio volatility is equal to the weighted sum of the asset volatilities:
      $\sigma_P=w_1\sigma_1+w_2\sigma_2$

Derivation of portfolio volatility:

- $cov(\tilde R_1,\tilde R_2)=\sigma(\tilde R_1)\sigma(\tilde R_2)$

- $\sigma_P^2=w_1^2\sigma_1^2+w_2^2\sigma_2^2+2w_1w_2\sigma_1\sigma_2=(w_1\sigma_1+w_2\sigma_2)^2$

- Therefore:

  $\sigma_P=w_1\sigma_1+w_2\sigma_2$

In [ ]:
# Define the correlation coefficient
corr_coef = 1


In [ ]:
print(f"Portfolio return: {100*calc_portfolio_return(p1, p2, ret1, ret2):.2f} %")
print(
    f"Portfolio volatility: {100*calc_portfolio_std(p1, p2, std1, std2, corr_coef):.2f} %"
)


We will vary the weights of the two assets and observe how portfolio return and volatility change.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

weights_1 = np.linspace(
    0, 1, 101
)  # Generate 101 points between 0 and 1 representing the weight of asset 1

portfolio_returns = []
portfolio_stds = []

for w1 in weights_1:
    w2 = 1 - w1

    ret_p = calc_portfolio_return(w1, w2, ret1, ret2)
    std_p = calc_portfolio_std(w1, w2, std1, std2, corr_coef)

    portfolio_returns.append(ret_p)
    portfolio_stds.append(std_p)

plt.scatter(portfolio_stds, portfolio_returns, marker=".", color="blue", alpha=0.5)

plt.scatter(std1, ret1, color="red", marker="*", s=200, label="Asset 1")
plt.scatter(std2, ret2, color="green", marker="*", s=200, label="Asset 2")

plt.xlabel("Volatility")
plt.ylabel("Return")
plt.title(r"Two-asset portfolios with correlation coefficient $\rho=1$")
plt.legend()
plt.show()


---

We will now build portfolios using the returns of two downloaded assets.

In [ ]:
# Download data from GitHub
file_path = "https://raw.githubusercontent.com/alfonso-santos/microcredencial-carteras-python/refs/heads/Feb-2025/Tema_1_Activos_Intro/data/precios_corr_pos.csv"
prices_pos_df = pd.read_csv(file_path, index_col=0, parse_dates=True)
prices_pos_df.columns = ["Asset 1", "Asset 2"]
prices_pos_df.head()


In [ ]:
# Load asset prices from a local file
prices_pos_df = pd.read_csv(
    "../data/precios_corr_pos.csv", index_col=0, parse_dates=True
)
prices_pos_df.columns = ["Asset 1", "Asset 2"]
prices_pos_df.head()


In [ ]:
# Calculate logarithmic returns
returns_pos_df = np.log(prices_pos_df).diff().dropna()


In [ ]:
# Plot prices and show the return correlation coefficient in the title
plt.figure(figsize=(10, 3))

corr_pos = returns_pos_df.corr().iloc[0, 1]

plt.plot(prices_pos_df["Asset 1"], label="Asset 1", color="blue")
plt.plot(prices_pos_df["Asset 2"], label="Asset 2", color="green")

plt.title(f"Positively Correlated Prices ($\\rho={corr_pos:.2f}$)")
plt.xlabel("Time")
plt.ylabel("Price")
plt.legend()
plt.grid(True)
plt.show()


As shown in the figure, a **positive correlation** means that when one asset rises, the other tends to rise as well, and vice versa. In other words, the two assets tend to move in the same direction.

---

We will generate 1,000 random portfolios and plot them.

In [ ]:
# Function to calculate return and risk for an individual asset
def asset_performance(returns):
    return returns.mean(), returns.std()


In [ ]:
# Function to calculate portfolio return and risk
def portfolio_performance(weights, returns):
    portfolio_returns = returns @ weights.T
    expected_portfolio_returns = portfolio_returns.mean(axis=0)

    cov_matrix = returns.cov()
    portfolio_risk = np.sqrt(np.diag(weights @ cov_matrix @ weights.T))

    return expected_portfolio_returns, portfolio_risk


In [ ]:
def portfolio_performance(weights, returns):
    portfolio_returns = returns @ weights.T
    portfolio_risk = portfolio_returns.std(axis=0)
    expected_portfolio_returns = portfolio_returns.mean(axis=0)

    return expected_portfolio_returns, portfolio_risk


In [ ]:
# Function to plot portfolios on a single axis
def plot_on_axis(
    ret_mean,
    ret_std,
    asset1_mean,
    asset1_std,
    asset2_mean,
    asset2_std,
    color,
    marker,
    label,
    corr,
):
    plt.scatter(ret_std, ret_mean, c=color, marker=marker, alpha=0.4, label=label)
    plt.scatter(asset1_std, asset1_mean, c="red", marker="*", s=150, label="Asset 1")
    plt.scatter(asset2_std, asset2_mean, c="green", marker="*", s=150, label="Asset 2")

    plt.title(f"{label} (Correlation: {corr:.2f})")
    plt.xlabel("Risk (Standard Deviation)")
    plt.ylabel("Return (Mean Return)")
    plt.legend()
    plt.grid(True)


In [ ]:
# Generate random weights
num_portfolios = 1000

rng = np.random.default_rng(seed=42)
weights = rng.random(size=(num_portfolios, 2))
weights = weights / np.sum(
    weights, axis=1, keepdims=True
)  # Normalize so that each row sums to 1

weights.shape


In [ ]:
asset1_pos_mean, asset1_pos_std = asset_performance(returns_pos_df["Asset 1"])
asset2_pos_mean, asset2_pos_std = asset_performance(returns_pos_df["Asset 2"])

ret_pos_mean, ret_pos_std = portfolio_performance(weights, returns_pos_df)

plt.figure(figsize=(14, 4))

plot_on_axis(
    ret_pos_mean,
    ret_pos_std,
    asset1_pos_mean,
    asset1_pos_std,
    asset2_pos_mean,
    asset2_pos_std,
    "blue",
    "o",
    "Portfolios",
    corr_pos,
)

plt.show()


---

### Case 2:  
$\quad \rho_{12}=-1$

* If $\rho_{12}=-1$, the assets are **perfectly negatively correlated**.

    - Diversification benefits are maximized.

    - Portfolio return is unchanged because it does not depend on the relationship between the assets:
      $E(\tilde R_P)=w_1E(\tilde R_1)+w_2E(\tilde R_2)$

    - Portfolio volatility is:
      $\sigma_P=|w_1\sigma_1-w_2\sigma_2|$

      The absolute value is necessary because volatility cannot be negative.

In [ ]:
# Define the correlation coefficient
corr_coef_neg = -1


In [ ]:
# Calculate portfolio return and volatility using the new correlation coefficient
print(f"Portfolio return: {100*calc_portfolio_return(p1, p2, ret1, ret2):.2f} %")
print(
    f"Portfolio volatility: {100*calc_portfolio_std(p1, p2, std1, std2, corr_coef_neg):.2f} %"
)


Remember that with $\rho_{12}=1$ we obtained:

* $E(\tilde R_p)=14$%
* $\sigma_p=19$%

With $\rho_{12}=-1$ we obtained:

* $E(\tilde R_p)=14$%
* $\sigma_p=1$%

Thanks to **diversification**, we obtain the same expected return with **lower volatility**.

In [ ]:
weights_1 = np.linspace(
    0, 1, 101
)  # Generate 101 points between 0 and 1 representing the weight of asset 1

portfolio_returns = []
portfolio_stds = []

for w1 in weights_1:
    w2 = 1 - w1

    ret_p = calc_portfolio_return(w1, w2, ret1, ret2)
    std_p = calc_portfolio_std(w1, w2, std1, std2, corr_coef_neg)

    portfolio_returns.append(ret_p)
    portfolio_stds.append(std_p)

fig, ax = plt.subplots(figsize=(12, 6))

plt.scatter(portfolio_stds, portfolio_returns, marker=".", color="blue", alpha=0.5)

plt.scatter(std1, ret1, color="red", marker="*", s=200, label="Asset 1")
plt.scatter(std2, ret2, color="green", marker="*", s=200, label="Asset 2")

plt.xlabel("Volatility")
plt.ylabel("Return")
plt.title(r"Two-asset portfolios with correlation coefficient $\rho=-1$")
plt.legend()
plt.show()


---

We will now build portfolios using the returns of two downloaded assets.

In [ ]:
file_path = "https://raw.githubusercontent.com/alfonso-santos/microcredencial-carteras-python/refs/heads/Feb-2025/Tema_1_Activos_Intro/data/precios_corr_neg.csv"
prices_neg_df = pd.read_csv(file_path, index_col=0, parse_dates=True)
prices_neg_df.columns = ["Asset 1", "Asset 2"]
prices_neg_df.head()


In [ ]:
prices_neg_df = pd.read_csv(
    "../data/precios_corr_neg.csv", index_col=0, parse_dates=True
)
prices_neg_df.columns = ["Asset 1", "Asset 2"]
prices_neg_df.head()


In [ ]:
# Calculate logarithmic returns
returns_neg_df = np.log(prices_neg_df).diff().dropna()


In [ ]:
# Plot prices and show the return correlation coefficient in the title
plt.figure(figsize=(10, 3))

corr_neg = returns_neg_df.corr().iloc[0, 1]

plt.plot(prices_neg_df["Asset 1"], label="Asset 1", color="blue")
plt.plot(prices_neg_df["Asset 2"], label="Asset 2", color="green")

plt.title(f"Negatively Correlated Prices ($\\rho={corr_neg:.2f}$)")
plt.xlabel("Time")
plt.ylabel("Price")
plt.legend()
plt.grid(True)
plt.show()


The figure clearly shows what **negative correlation** means. When one asset rises, the other tends to fall, so they move in opposite directions. We will now generate 1,000 random portfolios.

In [ ]:
# Generate random weights
num_portfolios = 1000

rng = np.random.default_rng(seed=42)
weights = rng.random(size=(num_portfolios, 2))
weights = weights / np.sum(
    weights, axis=1, keepdims=True
)  # Normalize so that each row sums to 1


In [ ]:
asset1_neg_mean, asset1_neg_std = asset_performance(returns_neg_df["Asset 1"])
asset2_neg_mean, asset2_neg_std = asset_performance(returns_neg_df["Asset 2"])

ret_neg_mean, ret_neg_std = portfolio_performance(weights, returns_neg_df)

plt.figure(figsize=(14, 4))

plot_on_axis(
    ret_neg_mean,
    ret_neg_std,
    asset1_neg_mean,
    asset1_neg_std,
    asset2_neg_mean,
    asset2_neg_std,
    "blue",
    "o",
    "Portfolios",
    corr_neg,
)

plt.show()


---

### Case 3: $\quad$ $-1\leq\rho_{12}\leq1$

* If $-1\leq\rho_{12}\leq1$, the assets have an **imperfect linear correlation**.

    - There are diversification benefits. In general, the lower the correlation, the greater the diversification benefit.

    - Portfolio return is unchanged because it does not depend on the relationship between the assets:
      $E(\tilde R_P)=w_1E(\tilde R_1)+w_2E(\tilde R_2)$

    - Portfolio variance is:
      $\sigma^2(\tilde R_p)=w_1^2\sigma^2(\tilde R_1)+w_2^2\sigma^2(\tilde R_2)+2w_1w_2\rho_{12}\sigma_1\sigma_2$

    - Portfolio volatility is the square root of the variance:
      $\sigma(\tilde R_p)=\sqrt{w_1^2\sigma^2(\tilde R_1)+w_2^2\sigma^2(\tilde R_2)+2w_1w_2\rho_{12}\sigma_1\sigma_2}$

In [ ]:
weights_1 = np.linspace(0, 1, 101)
corr_coef_list = [-1, -0.8, -0.5, 0, 0.5, 1]

colors = ["#440154", "#3b528b", "#21918c", "#5ec962", "#3366CC", "#FF6633"]

layout_width = 1200
layout_height = 700

fig = go.Figure()

for idx, corr_coef_i in enumerate(corr_coef_list):
    return_list = []
    std_list = []

    for w1 in weights_1:
        w2 = 1 - w1
        ret_p = calc_portfolio_return(w1, w2, ret1, ret2)
        std_p = calc_portfolio_std(w1, w2, std1, std2, corr_coef_i)

        return_list.append(ret_p)
        std_list.append(std_p)

    fig.add_trace(
        go.Scatter(
            x=std_list,
            y=return_list,
            mode="markers",
            name=f"Corr {corr_coef_i}",
            marker=dict(color=colors[idx], size=8, opacity=0.5),
        )
    )

fig.add_trace(
    go.Scatter(
        x=[std1],
        y=[ret1],
        mode="markers",
        marker=dict(color="red", size=20, symbol="star"),
        name="Asset 1",
    )
)

fig.add_trace(
    go.Scatter(
        x=[std2],
        y=[ret2],
        mode="markers",
        marker=dict(color="green", size=20, symbol="star"),
        name="Asset 2",
    )
)

fig.update_layout(
    title="Two-asset portfolios for different correlation coefficients",
    title_x=0.5,
    xaxis_title="Volatility",
    yaxis_title="Return",
    width=layout_width,
    height=layout_height,
)

fig.show()


---

We will now build portfolios using the returns of two downloaded assets.

In [ ]:
# Read data from GitHub
file_path = "https://raw.githubusercontent.com/alfonso-santos/microcredencial-carteras-python/refs/heads/Feb-2025/Tema_1_Activos_Intro/data/precios_corr_cero.csv"
prices_zero_df = pd.read_csv(file_path, index_col=0, parse_dates=True)
prices_zero_df.columns = ["Asset 1", "Asset 2"]
prices_zero_df.head()


In [ ]:
prices_zero_df = pd.read_csv(
    "../data/precios_corr_cero.csv", index_col=0, parse_dates=True
)
prices_zero_df.columns = ["Asset 1", "Asset 2"]
prices_zero_df.head()


In [ ]:
# Calculate logarithmic returns
returns_zero_df = np.log(prices_zero_df).diff().dropna()


In [ ]:
# Plot prices and show the return correlation coefficient in the title
plt.figure(figsize=(10, 3))

corr_zero = returns_zero_df.corr().iloc[0, 1]

plt.plot(prices_zero_df["Asset 1"], label="Asset 1", color="blue")
plt.plot(prices_zero_df["Asset 2"], label="Asset 2", color="green")

plt.title(f"Low-Correlation Prices ($\\rho={corr_zero:.2f}$)")
plt.xlabel("Time")
plt.ylabel("Price")
plt.legend()
plt.grid(True)
plt.show()


As shown in the figure, two assets have **low correlation** when there is little relationship between their movements. When one asset rises, we cannot systematically say whether the other will rise or fall. We will once again generate 1,000 random portfolios.

In [ ]:
# Generate random weights
num_portfolios = 1000

rng = np.random.default_rng(seed=42)
weights = rng.random(size=(num_portfolios, 2))
weights = weights / np.sum(
    weights, axis=1, keepdims=True
)  # Normalize so that each row sums to 1


In [ ]:
asset1_zero_mean, asset1_zero_std = asset_performance(returns_zero_df["Asset 1"])
asset2_zero_mean, asset2_zero_std = asset_performance(returns_zero_df["Asset 2"])

ret_zero_mean, ret_zero_std = portfolio_performance(weights, returns_zero_df)

plt.figure(figsize=(14, 4))

plot_on_axis(
    ret_zero_mean,
    ret_zero_std,
    asset1_zero_mean,
    asset1_zero_std,
    asset2_zero_mean,
    asset2_zero_std,
    "purple",
    "o",
    "Portfolios",
    corr_zero,
)

plt.show()


## 2. Calculating the minimum-volatility portfolio

We will calculate the minimum-variance portfolio:

* Start from the portfolio variance formula:

  $\sigma_p^2=w_1^2\sigma_1^2+w_2^2\sigma_2^2+2w_1w_2\rho_{12}\sigma_1\sigma_2$

* Substitute $w_2=1-w_1$, differentiate with respect to $w_1$, and set the derivative equal to zero:

  $\frac{\partial\sigma_p^2}{\partial w_1}=2w_1\sigma_1^2-2(1-w_1)\sigma_2^2+2(1-2w_1)\rho_{12}\sigma_1\sigma_2=0$

* Solving for $w_1$ gives:

  $w_1=\frac{\sigma_2^2-\rho_{12}\sigma_1\sigma_2}{\sigma_1^2+\sigma_2^2-2\rho_{12}\sigma_1\sigma_2}$

For two assets with correlation $\rho_{12}=-1$:

$w_1=\frac{\sigma_2^2+\sigma_1\sigma_2}{\sigma_1^2+\sigma_2^2+2\sigma_1\sigma_2}
=\frac{\sigma_2(\sigma_1+\sigma_2)}{(\sigma_1+\sigma_2)^2}
=\frac{\sigma_2}{\sigma_1+\sigma_2}$

Applying this result to the two-asset portfolio example we have been using:

In [ ]:
p1 = std2 / (std1 + std2)
p2 = 1 - p1

print(f"Weight of asset 1: {100*p1:.2f}%")
print(f"Weight of asset 2: {100*p2:.2f}%")


In [ ]:
# Calculate portfolio return and volatility using the new weights
print(f"Portfolio return: {100*calc_portfolio_return(p1, p2, ret1, ret2):.2f} %")
print(
    f"Portfolio volatility: {100*calc_portfolio_std(p1, p2, std1, std2, corr_coef_neg):.2f} %"
)


**CONCLUSIONS**

Remember that with the initial weights and $\rho_{12}=1$ we obtained:

* $E(\tilde R_p)=14$%
* $\sigma_p=19$%

With $\rho_{12}=-1$ and the minimum-variance portfolio formula we obtain:

* $E(\tilde R_p)=13.75$%
* $\sigma_p=0$%

Thanks to **diversification**, we obtain almost the same expected return with **no volatility** in this theoretical case.

## EXERCISE

Download data for Telefónica and Banco Santander for 2022 and 2023 and complete the following tasks:

1. Generate 100 portfolios by changing the weight of asset 1 in steps of 0.01. Plot all portfolios and the two individual assets in the return-risk plane.

2. Create the minimum-volatility portfolio and include it in the graph generated above.

3. Compare the return, volatility, skewness, and kurtosis of the two assets and the portfolio.

In [ ]:
assets = ["TEF", "SAN"]

prices = yf.download(
    assets,
    start="2022-01-01",
    end="2024-01-01",
    auto_adjust=False,
    multi_level_index=False,
)["Adj Close"]

prices.head()


In [ ]:
returns = np.log(prices).diff().dropna()
returns.head()


In [ ]:
weights_1 = np.linspace(0, 1, 101)

means = []
stds = []

for w1 in weights_1:
    w2 = 1 - w1
    weights = np.array([w1, w2])

    portfolio_returns = returns @ weights
    mean = portfolio_returns.mean()
    std = portfolio_returns.std()

    means.append(mean)
    stds.append(std)

means = np.array(means)
stds = np.array(stds)

plt.plot(stds * np.sqrt(252), means * 252, "-")
plt.xlabel("Volatility")
plt.ylabel("Return")
plt.show()


$w_1=\dfrac{\sigma_2^2-\rho_{12}\sigma_1\sigma_2}{\sigma_1^2+\sigma_2^2-2\rho_{12}\sigma_1\sigma_2}$

In [ ]:
# 2 - Create the minimum-volatility portfolio
std1 = returns.std()["SAN"]
std2 = returns.std()["TEF"]
rho = returns.corr().iloc[0, 1]

w1 = (std2**2 - rho * std1 * std2) / (
    std1**2 + std2**2 - 2 * rho * std1 * std2
)
w2 = 1 - w1

weights = np.array([w1, w2])
weights


In [ ]:
# 3 - Plot the two assets and the portfolio in the return-risk plane
portfolio_returns = returns @ weights

mean1 = returns.mean()["SAN"]
mean2 = returns.mean()["TEF"]
portfolio_mean = portfolio_returns.mean()
portfolio_std = portfolio_returns.std()

plt.plot(stds * np.sqrt(252), means * 252, "-")
plt.plot(std1 * np.sqrt(252), mean1 * 252, "o", label="SAN")
plt.plot(std2 * np.sqrt(252), mean2 * 252, "o", label="TEF")
plt.plot(
    portfolio_std * np.sqrt(252),
    portfolio_mean * 252,
    "o",
    color="black",
    label="Portfolio",
)

plt.title("Return-Risk of the Assets and the Portfolio")
plt.xlabel("Volatility (Risk)")
plt.ylabel("Return")
plt.legend()
plt.show()


In [ ]:
# 4 - Compare return, volatility, skewness, and kurtosis
returns_all = returns.copy()
returns_all["Portfolio"] = portfolio_returns
returns_all


In [ ]:
returns_all.mean() * 252


In [ ]:
returns_all.std() * np.sqrt(252)


In [ ]:
returns_all.skew()


In [ ]:
returns_all.kurtosis()
